# NYC TLC Trip Data - 20 Preguntas de Negocio (PSet 02)

Este notebook responde las 20 preguntas del enunciado usando exclusivamente `gold.*`.

In [2]:
import os
import pandas as pd
import psycopg2
port = int(os.getenv('POSTGRES_PORT', '5432'))
dbname = os.getenv('POSTGRES_DB', 'nyc_trips')
user = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')
if not user or not password:
    raise ValueError(
        "Faltan POSTGRES_USER/POSTGRES_PASSWORD en variables de entorno."
    )
hosts = [
    os.getenv('POSTGRES_HOST'),  # si viene definido, primero
    'localhost',                 # típico cuando corres notebook en tu PC
    'postgres',                  # típico dentro de contenedores
]
hosts = [h for h in hosts if h]  # limpia None/vacíos, mantiene orden
last_error = None
conn = None
for host in hosts:
    try:
        conn = psycopg2.connect(
            host=host,
            port=port,
            dbname=dbname,
            user=user,
            password=password,
        )
        print(f"Conectado a PostgreSQL en host={host}, db={dbname}")
        break
    except Exception as e:
        last_error = e
        print(f"No se pudo conectar con host={host}: {e}")
if conn is None:
    raise RuntimeError(f"No fue posible conectar a PostgreSQL. Ultimo error: {last_error}")
def run(sql):
    return pd.read_sql(sql, conn)

No se pudo conectar con host=postgres: could not translate host name "postgres" to address: Name or service not known

Conectado a PostgreSQL en host=localhost, db=nyc_trips


## 1) Viajes por mes (2024)
Tablas usadas: `gold.fct_trips`.
Interpretacion: La demanda presenta estacionalidad en 2024, con meses de mayor y menor volumen. Esto sugiere variaciones por clima, turismo y dinamica urbana.

In [8]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= '2024-01-01' AND pickup_date_key < '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_93996\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,trips
0,2024-01-01 00:00:00+00:00,2985433
1,2024-02-01 00:00:00+00:00,3024850
2,2024-03-01 00:00:00+00:00,3595437
3,2024-04-01 00:00:00+00:00,3526733
4,2024-05-01 00:00:00+00:00,3735589
5,2024-06-01 00:00:00+00:00,3544924
6,2024-07-01 00:00:00+00:00,3078133
7,2024-08-01 00:00:00+00:00,2977864
8,2024-09-01 00:00:00+00:00,3631133
9,2024-10-01 00:00:00+00:00,3828314


## 2) Viajes por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El servicio yellow concentra mas viajes que green en todos los meses, aunque ambos siguen una estacionalidad parecida. La diferencia principal es de escala.

In [2]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, COUNT(*) AS trips
FROM gold.fct_trips
GROUP BY 1, 2
ORDER BY 1, 2;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,service_type,trips
0,2022-01-01 00:00:00+00:00,green,62300
1,2022-01-01 00:00:00+00:00,yellow,2449613
2,2022-02-01 00:00:00+00:00,green,69199
3,2022-02-01 00:00:00+00:00,yellow,2962240
4,2022-03-01 00:00:00+00:00,green,78309
...,...,...,...
90,2025-10-01 00:00:00+00:00,green,49286
91,2025-10-01 00:00:00+00:00,yellow,4319971
92,2025-11-01 00:00:00+00:00,green,46759
93,2025-11-01 00:00:00+00:00,yellow,4053118


## 3) Top 10 zonas de pickup (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Los pickups se concentran en hubs de alta demanda (aeropuertos y zonas centrales). Esto confirma una centralizacion geografica del origen de viajes.

In [3]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone_name,borough,trips
0,JFK Airport,Queens,1910244
1,Upper East Side South,Manhattan,1892772
2,Midtown Center,Manhattan,1886880
3,Upper East Side North,Manhattan,1718376
4,Midtown East,Manhattan,1399892
5,Times Sq/Theatre District,Manhattan,1362178
6,Penn Station/Madison Sq West,Manhattan,1340585
7,Lincoln Square East,Manhattan,1299332
8,LaGuardia Airport,Queens,1275003
9,Murray Hill,Manhattan,1160466


## 4) Top 10 zonas de dropoff (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Los dropoffs mas frecuentes tambien se ubican en zonas de alta actividad. El patron de destinos es consistente con los origenes mas demandados.

In [4]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.do_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone_name,borough,trips
0,Upper East Side North,Manhattan,1807550
1,Upper East Side South,Manhattan,1714789
2,Midtown Center,Manhattan,1521470
3,Times Sq/Theatre District,Manhattan,1286410
4,Murray Hill,Manhattan,1199480
5,Midtown East,Manhattan,1171128
6,Lincoln Square East,Manhattan,1137196
7,Upper West Side South,Manhattan,1135547
8,East Chelsea,Manhattan,1063507
9,Lenox Hill West,Manhattan,1058186


## 5) Top 5 boroughs por mes (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Manhattan domina de forma recurrente y Queens destaca por flujo aeroportuario. La composicion mensual por borough es estable en estructura general.

In [6]:
run("""
WITH base AS (
  SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough, COUNT(*) AS trips
  FROM gold.fct_trips f
  JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
  WHERE f.pickup_date_key >= DATE '2024-01-01'
    AND f.pickup_date_key < DATE '2025-01-01'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT month, borough, trips
FROM r
WHERE rn <= 5
ORDER BY month, trips DESC;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,borough,trips
0,2024-01-01 00:00:00+00:00,Manhattan,2651102
1,2024-01-01 00:00:00+00:00,Queens,281604
2,2024-01-01 00:00:00+00:00,Brooklyn,32709
3,2024-01-01 00:00:00+00:00,Unknown,10379
4,2024-01-01 00:00:00+00:00,Bronx,7719
5,2024-02-01 00:00:00+00:00,Manhattan,2712722
6,2024-02-01 00:00:00+00:00,Queens,257742
7,2024-02-01 00:00:00+00:00,Brooklyn,35309
8,2024-02-01 00:00:00+00:00,Unknown,9695
9,2024-02-01 00:00:00+00:00,Bronx,7641


## 5.1) Top 5 boroughs por mes (pickup) excluyendo `Unknown`
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: para lectura de negocio geografica, se excluye `Unknown`; este valor se considera un indicador de calidad de datos y puede analizarse por separado.

In [3]:
run("""
WITH base AS (
  SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough, COUNT(*) AS trips
  FROM gold.fct_trips f
  JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
  WHERE f.pickup_date_key >= DATE '2024-01-01'
    AND f.pickup_date_key < DATE '2025-01-01'
    AND z.borough <> 'Unknown'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT month, borough, trips
FROM r
WHERE rn <= 5
ORDER BY month, trips DESC;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_82648\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,borough,trips
0,2024-01-01 00:00:00+00:00,Manhattan,2651102
1,2024-01-01 00:00:00+00:00,Queens,281604
2,2024-01-01 00:00:00+00:00,Brooklyn,32709
3,2024-01-01 00:00:00+00:00,Bronx,7719
4,2024-01-01 00:00:00+00:00,EWR,287
5,2024-02-01 00:00:00+00:00,Manhattan,2712722
6,2024-02-01 00:00:00+00:00,Queens,257742
7,2024-02-01 00:00:00+00:00,Brooklyn,35309
8,2024-02-01 00:00:00+00:00,Bronx,7641
9,2024-02-01 00:00:00+00:00,EWR,281


## 6) Horas pico (top 5 horas) para cada dia de semana
Tablas usadas: `gold.fct_trips`.
Interpretacion: Se observan horas pico por dia de semana, utiles para planificar oferta operativa. El patron refleja commuting y actividad de ocio.

In [7]:
run("""
WITH base AS (
  SELECT pickup_day_of_week, pickup_hour, COUNT(*) AS trips
  FROM gold.fct_trips
  WHERE pickup_date_key >= DATE '2024-01-01'
    AND pickup_date_key < DATE '2025-01-01'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY pickup_day_of_week ORDER BY trips DESC) AS rn
  FROM base
)
SELECT pickup_day_of_week, pickup_hour, trips
FROM r
WHERE rn <= 5
ORDER BY pickup_day_of_week, trips DESC;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,pickup_day_of_week,pickup_hour,trips
0,0,14,336032
1,0,17,331945
2,0,16,330784
3,0,13,325825
4,0,15,322741
5,1,18,373005
6,1,17,367109
7,1,15,340957
8,1,16,336325
9,1,14,330361


## 7) Distribucion de viajes por dia de semana (ranking)
Tablas usadas: `gold.fct_trips`.
Interpretacion: Hay dias sistematicamente mas activos que otros, lo que evidencia un ciclo semanal claro. Este ranking sirve para ajustar capacidad por dia.

In [8]:
run("""
SELECT pickup_day_of_week, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY trips DESC;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,pickup_day_of_week,trips
0,4,6445406
1,6,6218745
2,3,6136158
3,5,6111228
4,2,5916522
5,0,5261268
6,1,5127489


## 8) Ingreso total (`total_amount`) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El ingreso mensual sigue de cerca el volumen de viajes. Los meses con mayor demanda tienden a mayor recaudacion total.

In [9]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,total_revenue
0,2024-01-01 00:00:00+00:00,8.161710e+07
1,2024-02-01 00:00:00+00:00,8.220750e+07
2,2024-03-01 00:00:00+00:00,9.965075e+07
3,2024-04-01 00:00:00+00:00,9.912941e+07
4,2024-05-01 00:00:00+00:00,1.085226e+08
5,2024-06-01 00:00:00+00:00,1.015710e+08
6,2024-07-01 00:00:00+00:00,8.918056e+07
7,2024-08-01 00:00:00+00:00,8.715687e+07
8,2024-09-01 00:00:00+00:00,1.067667e+08
9,2024-10-01 00:00:00+00:00,1.121499e+08


## 9) Ingreso total por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: Yellow aporta la mayor parte del ingreso en cada mes, consistente con su mayor cantidad de viajes. Green complementa en menor escala.

In [10]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,service_type,total_revenue
0,2024-01-01 00:00:00+00:00,green,1.268964e+06
1,2024-01-01 00:00:00+00:00,yellow,8.034814e+07
2,2024-02-01 00:00:00+00:00,green,1.214659e+06
3,2024-02-01 00:00:00+00:00,yellow,8.099284e+07
4,2024-03-01 00:00:00+00:00,green,1.318139e+06
5,2024-03-01 00:00:00+00:00,yellow,9.833261e+07
6,2024-04-01 00:00:00+00:00,green,1.322417e+06
7,2024-04-01 00:00:00+00:00,yellow,9.780699e+07
8,2024-05-01 00:00:00+00:00,green,1.501840e+06
9,2024-05-01 00:00:00+00:00,yellow,1.070207e+08


## 10) Tip % promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El tip porcentual promedio cambia por mes, lo que sugiere variaciones de comportamiento de pago y contexto operativo.

In [11]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month,
       AVG(tip_amount / NULLIF(fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
  AND fare_amount > 0
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,avg_tip_pct
0,2024-01-01 00:00:00+00:00,0.229606
1,2024-02-01 00:00:00+00:00,0.209194
2,2024-03-01 00:00:00+00:00,0.196349
3,2024-04-01 00:00:00+00:00,0.200497
4,2024-05-01 00:00:00+00:00,0.201564
5,2024-06-01 00:00:00+00:00,0.191529
6,2024-07-01 00:00:00+00:00,0.194172
7,2024-08-01 00:00:00+00:00,0.188166
8,2024-09-01 00:00:00+00:00,0.189097
9,2024-10-01 00:00:00+00:00,0.196284


## 11) Tip % por borough y mes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Existen diferencias de tip porcentual entre boroughs, lo que indica perfiles de demanda y patrones de pago distintos por zona.

In [12]:
run("""
SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.fare_amount > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,borough,avg_tip_pct
0,2024-01-01 00:00:00+00:00,Bronx,0.024663
1,2024-01-01 00:00:00+00:00,Brooklyn,0.094015
2,2024-01-01 00:00:00+00:00,EWR,0.477636
3,2024-01-01 00:00:00+00:00,Manhattan,0.217086
4,2024-01-01 00:00:00+00:00,Queens,0.209656
...,...,...,...
91,2024-12-01 00:00:00+00:00,Manhattan,0.204972
92,2024-12-01 00:00:00+00:00,Queens,0.160702
93,2024-12-01 00:00:00+00:00,Staten Island,0.250454
94,2024-12-01 00:00:00+00:00,Unknown,0.198617


## 12) Top 10 zonas por ingreso total (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las zonas con mayor ingreso total se alinean con areas de alto flujo. No siempre coinciden exactamente con las de mayor volumen, por ticket promedio.

In [13]:
run("""
SELECT z.zone_name, z.borough, SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY total_revenue DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone_name,borough,total_revenue
0,JFK Airport,Queens,1.569750e+08
1,LaGuardia Airport,Queens,8.722304e+07
2,Midtown Center,Manhattan,4.767217e+07
3,Times Sq/Theatre District,Manhattan,3.969221e+07
4,Upper East Side South,Manhattan,3.959266e+07
5,Upper East Side North,Manhattan,3.629119e+07
6,Midtown East,Manhattan,3.444941e+07
7,Penn Station/Madison Sq West,Manhattan,3.433714e+07
8,Midtown North,Manhattan,2.919408e+07
9,Lincoln Square East,Manhattan,2.905842e+07


## 13) Top 10 zonas por tip % (pickup) con minimo N viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Con un minimo de viajes, el ranking de tip% es mas robusto y menos sensible a outliers. Se identifican zonas con propina alta de forma consistente.

In [14]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.fare_amount > 0
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY tip_pct DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone_name,borough,trips,tip_pct
0,Outside of NYC,None,22711,5.893409
1,Newark Airport,EWR,5774,2.064473
2,Williamsbridge/Olinville,Bronx,4259,1.489954
3,South Ozone Park,Queens,16294,0.942148
4,Baisley Park,Queens,15279,0.767359
5,Red Hook,Brooklyn,10715,0.522015
6,Bedford,Brooklyn,14358,0.383915
7,Melrose South,Bronx,6589,0.306622
8,Forest Hills,Queens,42915,0.266601
9,Steinway,Queens,15435,0.265156


## 14) Comparacion cash vs card: viajes, ingreso total, tip %
Tablas usadas: `gold.fct_trips`, `gold.dim_payment_type`.
Interpretacion: Card y cash muestran diferencias en volumen, ingreso y propina. En general, card tiende a mejor tip porcentual promedio.

In [15]:
run("""
SELECT p.payment_type, COUNT(*) AS trips, SUM(f.total_amount) AS total_revenue,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_payment_type p ON f.payment_type_id = p.payment_type_id
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND p.payment_type IN ('cash', 'card')
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,payment_type,trips,total_revenue,avg_tip_pct
0,card,30905683,9.223864e+08,0.261360
1,cash,5567976,1.379138e+08,0.000014


## 15) Duracion promedio (min) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: La duracion media de viaje varia por mes, posiblemente por congestion, clima y mezcla de rutas.

In [16]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_duration_minutes) AS avg_duration_min
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,avg_duration_min
0,2024-01-01 00:00:00+00:00,15.705098
1,2024-02-01 00:00:00+00:00,16.068028
2,2024-03-01 00:00:00+00:00,16.734794
3,2024-04-01 00:00:00+00:00,17.104964
4,2024-05-01 00:00:00+00:00,18.101228
5,2024-06-01 00:00:00+00:00,17.642541
6,2024-07-01 00:00:00+00:00,17.286830
7,2024-08-01 00:00:00+00:00,17.449330
8,2024-09-01 00:00:00+00:00,18.692773
9,2024-10-01 00:00:00+00:00,18.343648


## 16) Distancia promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: La distancia promedio mensual cambia con la demanda y tipo de trayectos predominantes. Complementa el analisis de duracion.

In [17]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_distance) AS avg_distance
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,month,avg_distance
0,2024-01-01 00:00:00+00:00,4.186944
1,2024-02-01 00:00:00+00:00,4.115018
2,2024-03-01 00:00:00+00:00,4.676122
3,2024-04-01 00:00:00+00:00,5.415792
4,2024-05-01 00:00:00+00:00,5.544076
5,2024-06-01 00:00:00+00:00,5.380544
6,2024-07-01 00:00:00+00:00,5.237329
7,2024-08-01 00:00:00+00:00,5.216554
8,2024-09-01 00:00:00+00:00,5.963979
9,2024-10-01 00:00:00+00:00,5.239993


## 17) Velocidad promedio (mph) por borough y franja horaria
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: La velocidad promedio difiere por borough y franja horaria, reflejando efectos de trafico y densidad urbana.

In [18]:
run("""
SELECT z.borough,
       CASE WHEN f.pickup_hour BETWEEN 6 AND 11 THEN 'morning'
            WHEN f.pickup_hour BETWEEN 12 AND 17 THEN 'afternoon'
            WHEN f.pickup_hour BETWEEN 18 AND 23 THEN 'evening'
            ELSE 'night' END AS time_band,
       AVG(f.trip_distance / NULLIF(f.trip_duration_minutes / 60.0, 0)) AS avg_speed_mph
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.trip_duration_minutes > 0 AND f.trip_distance > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,borough,time_band,avg_speed_mph
0,Bronx,afternoon,109.386405
1,Bronx,evening,44.005565
2,Bronx,morning,155.150551
3,Bronx,night,76.637078
4,Brooklyn,afternoon,77.512370
5,Brooklyn,evening,38.739125
6,Brooklyn,morning,73.129060
7,Brooklyn,night,55.404808
8,EWR,afternoon,199.249341
9,EWR,evening,119.842516


## 18) Percentiles p50 y p90 de duracion por borough
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: El p50 representa el viaje tipico y el p90 muestra la cola de viajes largos. La brecha entre ambos indica dispersion de duraciones.

In [19]:
run("""
SELECT z.borough,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,borough,p50_duration,p90_duration
0,Bronx,27.630,71.020
1,Brooklyn,22.230,55.800
2,EWR,0.130,1.330
3,Manhattan,12.020,26.820
4,Queens,33.150,62.720
5,Staten Island,18.455,48.447
6,Unknown,12.350,34.300
7,None,0.350,31.230


## 19) Top 10 zonas (pickup) por p90 de duracion
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las zonas con mayor p90 concentran viajes extremos en tiempo. Son candidatas para monitoreo de congestion y planificacion operativa.

In [20]:
run("""
SELECT z.zone_name, z.borough,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY p90_duration DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone_name,borough,p90_duration
0,Far Rockaway,Queens,105.420
1,Hammels/Arverne,Queens,103.877
2,Coney Island,Brooklyn,100.285
3,Rockaway Park,Queens,94.540
4,Brighton Beach,Brooklyn,92.676
5,Rosedale,Queens,89.500
6,Gravesend,Brooklyn,89.208
7,Co-Op City,Bronx,88.104
8,Cambria Heights,Queens,88.038
9,Laurelton,Queens,87.310


## 20) Top 10 rutas borough->borough por numero de viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las rutas borough->borough mas frecuentes revelan los principales corredores de movilidad. Sirven para priorizar analisis y estrategias de servicio.

In [22]:
run("""
SELECT pu.borough AS pickup_borough, dz.borough AS dropoff_borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone pu ON f.pu_zone_key = pu.zone_key
JOIN gold.dim_zone dz ON f.do_zone_key = dz.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,pickup_borough,dropoff_borough,trips
0,Manhattan,Manhattan,34079887
1,Queens,Manhattan,2140376
2,Manhattan,Queens,1097881
3,Queens,Queens,1020826
4,Manhattan,Brooklyn,751498
5,Queens,Brooklyn,564231
6,Brooklyn,Brooklyn,386479
7,Brooklyn,Manhattan,206411
8,Manhattan,Bronx,126308
9,Queens,None,110268


## Evidencia de Partition Pruning
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: demostrar que el planner no escanea todas las particiones.

In [23]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM gold.fct_trips
WHERE pickup_date_key BETWEEN '2024-03-01' AND '2024-03-31';
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,QUERY PLAN
0,Finalize Aggregate (cost=114463.05..114463.06...
1,Buffers: shared hit=640 read=86606
2,-> Gather (cost=114462.83..114463.04 rows=...
3,Workers Planned: 2
4,Workers Launched: 2
5,Buffers: shared hit=640 read=86606
6,-> Partial Aggregate (cost=113462.83...
7,Buffers: shared hit=640 read=86606
8,-> Parallel Seq Scan on fct_tri...
9,Filter: ((pickup_date_key ...


In [24]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM gold.dim_zone
WHERE zone_key = 132;
""")

C:\Users\Tony\AppData\Local\Temp\ipykernel_40680\1087842843.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,QUERY PLAN
0,Seq Scan on dim_zone_p3 dim_zone (cost=0.00.....
1,Filter: (zone_key = 132)
2,Rows Removed by Filter: 48
3,Buffers: shared hit=1
4,Planning Time: 0.072 ms
5,Execution Time: 0.027 ms


In [25]:
conn.close()